# Monte Carlo Pass Search — one-match Colab demo
This notebook trains the paper's SMART, Player-to-Touch, Ball-at-Touch, and possession-value components on capped chronological subsets of one public Bundesliga match. It then runs leakage-free local/global counterfactual pass search. The outputs demonstrate the pipeline and compute cost; they are not calibrated tactical rankings.

In [ ]:
import os, subprocess, sys, pathlib
print(subprocess.run(['nvidia-smi'], text=True, capture_output=True).stdout or 'No NVIDIA GPU detected')
assert subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0, 'Select Runtime > Change runtime type > GPU'

In [ ]:
# Set this to the Git repository containing the small_demo implementation.
REPO_URL = ''
REPO_DIR = pathlib.Path('/content/monte-carlo-custom')
if not REPO_DIR.exists():
    assert REPO_URL, 'Set REPO_URL, then rerun this cell'
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    # Colab storage survives notebook reruns, so refresh an existing checkout.
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
required = pathlib.Path('monteCarloPassSearch/ballPlayerTrajModel/publicData/kloppy_to_preprocessed.py')
assert required.is_file(), f'Incomplete/stale checkout: missing {required}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'small_demo/requirements-colab.txt'], check=True)

## Prepare one match
Downloads the three J03WN1 files, creates chronological 70/15/15 splits, builds SMART vocabularies, extracts component datasets, and applies deterministic caps.

In [ ]:
def run_pipeline_stage(stage):
    result = subprocess.run([sys.executable, '-u', 'small_demo/run_pipeline.py', stage])
    if result.returncode:
        report_path = pathlib.Path('artifacts/mcps_small/compute_report.json')
        if report_path.exists():
            report = __import__('json').loads(report_path.read_text())
            failures = [row for row in report.get('stages', []) if row.get('return_code')]
            if failures:
                failed = failures[-1]
                log_path = pathlib.Path(failed['log'])
                print(f"\nFailed stage: {failed['name']}\nFull log: {log_path}")
                if log_path.exists(): print('\n'.join(log_path.read_text(errors='replace').splitlines()[-80:]))
        raise RuntimeError(f"Pipeline stage group {stage!r} failed; see the diagnostic above")

run_pipeline_stage('prepare')

## Train the four learned components
Architecture sizes and objectives match the supplied research implementation. Data and epoch counts are capped.

In [ ]:
subprocess.run([sys.executable, 'small_demo/run_pipeline.py', 'train'], check=True)

## Counterfactual search and reports
Each proposed ball flight conditions its own SMART player rollout. The default evaluates five held-out clips with 16 local and 16 global variants.

In [ ]:
subprocess.run([sys.executable, 'small_demo/run_pipeline.py', 'search'], check=True)
subprocess.run([sys.executable, 'small_demo/run_pipeline.py', 'report'], check=True)

In [ ]:
from IPython.display import display, Image
import json, pandas as pd
work = pathlib.Path('artifacts/mcps_small')
clip_file = work/'search/clip_percentiles.csv'
if clip_file.exists() and clip_file.stat().st_size > 1:
    try: display(pd.read_csv(clip_file).head())
    except pd.errors.EmptyDataError: print('No pass met the strict kick-fit criterion; inspect variants.csv diagnostics.')
for image in sorted((work/'figures').glob('*.png')): display(Image(filename=str(image)))
print(json.dumps(json.loads((work/'compute_report.json').read_text()), indent=2)[:12000])

## Interpretation
Inspect fit errors and model metrics before the pass values. With one match and capped training, useful evidence is that the architecture trains, candidate passes cause distinct generated futures, and compute stays within the measured budget. Do not treat the resulting passer order as an ability estimate.